## What if you convert your standard MongoDB query into natural language?

With an agentic multi-step workflow, tool calling, and ReAct (Reason and Act), this becomes possible:

1. The user asks something like: 'Give me the followers of person X.'

2. The LLM reasons: 'The user is asking for person X's follower details, which means I first need to search for person X.'

3. It calls a tool (a Python function responsible for searching the database by name or other parameters).

4. After fetching that particular user, it reasons again: 'It is confirmed that person X exists in the database. Now let me search for their followers using the $in operator.'

5. It calls the search tool again, this time querying with the $in operator for the follower IDs.

6. It responds with the details of the followers in natural language.

Get the MongoDB URI for the database you want to connect the agent to, and put it inside your .env file.

Import dependencies

In [2]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

load_dotenv()

True

In [ ]:
def get_db(db_name):
    uri = os.getenv("TOURDB_URI") #the uri
    client = MongoClient(uri)
    return client[db_name]

In [4]:
db = get_db("tourappdb")#the database

We want our agent to generate this part of the code.

In [5]:
users_cursor = db["users"].find({"number": 2342342342})

In [6]:
for user in users_cursor:
    print(user)

{'_id': ObjectId('6a685a2914384e4357013b58'), 'name': 'roxian', 'number': 2342342342.0, 'hikes': [ObjectId('6a68580b7e815fe7ce6ae28e'), {'hike': ObjectId('6a684826269d00f70a49aa5b'), 'booked': True, 'paid': True, '_id': ObjectId('6a70b1bc622cf93e4054cd49')}, {'hike': ObjectId('6a70b3162c5a36becb0c86e8'), 'booked': True, 'paid': True, '_id': ObjectId('6a70b43f2c5a36becb0c86f4')}], 'experience': 'Beginner', 'follower': [ObjectId('6a685a3b14384e4357013b59')], 'following': [ObjectId('6a685a3b14384e4357013b59'), ObjectId('6a686b95db4874428e179806'), ObjectId('6a70b34f2c5a36becb0c86ed')], 'posts': ['first post', 'second post'], 'createdAt': datetime.datetime(2026, 7, 28, 7, 28, 41, 73000), 'updatedAt': datetime.datetime(2026, 8, 3, 15, 44, 55, 981000), '__v': 0}


Import dependencies

In [8]:
import os

from dotenv import load_dotenv

from openai import OpenAI

from openai.types.responses import ResponseTextDeltaEvent

from agents import Agent, Runner, function_tool, SQLiteSession, RunConfig

from agents.models.openai_provider import OpenAIProvider

import wikipedia

import webbrowser

import urllib.parse

load_dotenv(override=True)

True

In [9]:
provider = OpenAIProvider(
api_key=os.getenv("GROQ_API_KEY"),
base_url="https://api.groq.com/openai/v1",
use_responses=False
)

In [10]:
agent = Agent(name="database agent",instructions="You are an agent with access to the MongoDB database. Your job is to write Python code (e.g., 'users_cursor = db['users'].find({'number': 2342342342})') to find specific users or objects. You can also perform CRUD operations. Write exact PyMongo code based on the user's prompt. Ensure names are in lowercase. Provide only the code snippet, without comments, as it will be executed directly via exec().",model="openai/gpt-oss-20b")

In [11]:
result= await Runner.run(agent,input("what you wanna do with the database"),run_config=RunConfig(model_provider=provider))

OPENAI_API_KEY is not set, skipping trace export


In [12]:
print(result.final_output)

users_cursor = db['users'].find({'name': {'$regex': 'sunilu', '$options': 'i'}})


## ! WARNING: We are about to run the code that the LLM generated, which is not a safe practice!

In [ ]:
exec (result.final_output)

In [ ]:
print("database updated")

In [13]:
for user in users_cursor:
    print(user)

This works, but it is not an agentic workflow. It isn't smart and carries a huge security risk of executing malicious Python code via exec().

### Let's build an agent with tool-calling capabilities so it can run specific tools to discover the database schema and safely decide which commands to run.

Create a session for conversation memory.

In [14]:
session = SQLiteSession(session_id="new_session_1")


Configure the model if you are using a provider other than OpenAI.

In [15]:
modelconfig=RunConfig(model_provider=provider)

Now, let's write some tools.

In [16]:
from agents import function_tool
@function_tool(strict_mode=False)
def query_users(query_dict: dict) -> str: # <- changed to str
    """Use this tool to find users in the database. Pass a dictionary query."""
    return str(list(db["users"].find(query_dict)))

@function_tool(strict_mode=False)
def update_user(query_dict: dict, update_dict: dict) -> str:
    """Use this tool to update a user. Pass the search query and the $set update."""
    result = db["users"].update_one(query_dict, update_dict)
    return f"Modified {result.modified_count} documents."

@function_tool(strict_mode=False)
def get_collections() -> str: # <- changed to str
    """Returns a list of all collections in the database."""
    return str(db.list_collection_names())
    
@function_tool(strict_mode=False)
def get_schema_sample(collection_name: str) -> str: # <- changed to str
    """Returns one document from a collection so you can see its structure."""
    return str(db[collection_name].find_one())


These tools are responsible for CRUD operations across the database and its collections.

Now pass the tools to the agent.

In [17]:
# Now pass these tools to your agent:
agent1 = Agent(
    name="database agent",
    instructions="You are a database assistant. Use your tools to answer the user's questions.",
    tools=[query_users, update_user,get_collections,get_schema_sample],
    model="openai/gpt-oss-20b"
)

Run the agent using Runner.run.

In [18]:
result=await Runner.run(
    agent1,
    input("what do you want to do from the database"),
    run_config=modelconfig,
    session=session
)

print(result.final_output)

**Search Result – User “sunil” or similar**

| Field | Value |
|-------|-------|
| **_id** | `6a685a3b14384e4357013b59` |
| **name** | `sunilu` |
| **number** | `23423423555.0` |
| **hikes** | `[]` |
| **experience** | `Expert` |
| **follower** | `[6a685a2914384e4357013b58, 6a686b95db4874428e179806, 6a70b34f2c5a36becb0c86ed]` |
| **following** | `[6a685a2914384e4357013b58, 6a686b95db4874428e179806, 6a70b34f2c5a36becb0c86ed]` |
| **posts** | `[]` |
| **createdAt** | `2026-07-28 07:28:59.117000` |
| **updatedAt** | `2026-08-03 16:01:00.676000` |
| **__v** | `0` |

**Interpretation**

- The database contains a single user whose name matches the pattern “sunil” (case‑insensitive regex). The closest match found is **“sunilu”**.
- No additional users were returned that closely match “sunil”.

**Next Steps**

- If you need more details about this user (e.g., follower/following relationships, posts, hikes, etc.), let me know and I can fetch related collections.
- If you want to search for othe

OPENAI_API_KEY is not set, skipping trace export


Now the Agentic AI kicks in! It's smart, natural, maintains the context of previous chats, and has the capability to reason through the user's prompt step-by-step.